<a href="https://colab.research.google.com/github/inmira/Data_Quality/blob/main/Data_Quality_Review_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Quality Review

Before building any features or models, I check the data carefully.

My goal here is to answer one question: can I trust this data enough to build reliable features on top of it?

I check completeness, consistency between tables, dates, and value ranges — before I move to feature engineering.

In [1]:
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression


## Step 1 - Load the data

The data comes as a zip file with three tables: customers, customer_snapshot, and transactions.

I unzip it first, then check what files are actually inside, before loading anything.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving data-20260922T122930Z-1-001.zip to data-20260922T122930Z-1-001.zip


In [3]:
import zipfile

with zipfile.ZipFile('data-20260922T122930Z-1-001.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

import os
print(os.listdir('data'))

['data']


In [4]:
print(os.listdir('.'))

['.config', 'data', 'data-20260922T122930Z-1-001.zip', 'sample_data']


## Step 2 - Load each table and check its structure

For each table, I look at the first few rows, and check column types and row counts.

This tells me what each table actually contains, before I compare them to each other.

In [5]:
customers_df = pd.read_csv('/content/data/data/customers.csv', encoding='iso-8859-1')
display(customers_df.head())

,customer_id,market,join_date,age_band,account_type,acquisition_channel,consent_marketing
0,C000001,DE,2023-09-11,25-34,standard,branch,True
1,C000002,PL,2024-11-11,25-34,standard,branch,True
2,C000003,DE,2019-03-30,18-24,basic,organic,True
3,C000004,GB,2019-07-15,55-64,premium,organic,True
4,C000005,GB,2022-08-11,55-64,standard,branch,True


In [6]:
customers_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   customer_id          2400 non-null   object
 1   market               2400 non-null   object
 2   join_date            2400 non-null   object
 3   age_band             2400 non-null   object
 4   account_type         2400 non-null   object
 5   acquisition_channel  2400 non-null   object
 6   consent_marketing    2400 non-null   bool  
dtypes: bool(1), object(6)
memory usage: 115.0+ KB


In [7]:
customer_snapshot_df = pd.read_csv('/content/data/data/customer_snapshot.csv', encoding='iso-8859-1')
display(customer_snapshot_df.head())

,customer_id,snapshot_date,last_app_login,balance_eur,overdraft_utilization,support_contacts_30d,declined_txn_30d,salary_in_90d,active_days_30d,txn_count_30d,txn_count_prev_30d,spend_30d,category_diversity_90d,churned_next_60d
0,C000001,2026-06-30,2026-06-28,2551.53,0.362,1,0,True,22,42,42,5199.52,14,False
1,C000002,2026-06-30,2026-06-16,3152.29,0.122,0,0,True,24,60,39,5898.63,12,False
2,C000003,2026-06-30,2026-06-25,5062.04,0.083,2,1,True,22,42,29,9244.64,13,False
3,C000004,2026-06-30,2026-06-30,4749.60,0.506,0,0,True,24,46,44,5694.04,14,False
4,C000005,2026-06-30,2026-03-14,1575.49,0.092,1,1,False,7,8,7,1094.47,10,True


In [8]:
customer_snapshot_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             2400 non-null   object 
 1   snapshot_date           2400 non-null   object 
 2   last_app_login          2400 non-null   object 
 3   balance_eur             2400 non-null   float64
 4   overdraft_utilization   2400 non-null   float64
 5   support_contacts_30d    2400 non-null   int64  
 6   declined_txn_30d        2400 non-null   int64  
 7   salary_in_90d           2400 non-null   bool   
 8   active_days_30d         2400 non-null   int64  
 9   txn_count_30d           2400 non-null   int64  
 10  txn_count_prev_30d      2400 non-null   int64  
 11  spend_30d               2400 non-null   float64
 12  category_diversity_90d  2400 non-null   int64  
 13  churned_next_60d        2400 non-null   bool   
dtypes: bool(2), float64(3), int64(6), object

In [9]:
transactions_df = pd.read_csv('/content/data/data/transactions.csv', encoding='iso-8859-1')
display(transactions_df.head())

,transaction_id,customer_id,booked_at,direction,amount_eur,merchant_category,merchant_name,channel,is_recurring,is_card_present,country_code
0,T000000010,C000001,2026-01-01,debit,-59.18,shopping,Shopping_05,transfer,False,False,DE
1,T000000130,C000001,2026-01-01,debit,-56.56,health,Health_08,transfer,False,False,DE
2,T000000429,C000002,2026-01-01,debit,-16.03,restaurants,Restaurants_26,card,False,True,PL
3,T000000446,C000002,2026-01-01,debit,-41.80,restaurants,Restaurants_28,card,False,True,PL
4,T000000579,C000003,2026-01-01,debit,-54.51,health,Health_13,transfer,False,False,DE


In [10]:
transactions_df.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 585823 entries, 0 to 585822
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   transaction_id     585823 non-null  object 
 1   customer_id        585823 non-null  object 
 2   booked_at          585823 non-null  object 
 3   direction          585823 non-null  object 
 4   amount_eur         585823 non-null  float64
 5   merchant_category  585703 non-null  object 
 6   merchant_name      585823 non-null  object 
 7   channel            585823 non-null  object 
 8   is_recurring       585823 non-null  bool   
 9   is_card_present    585823 non-null  bool   
 10  country_code       585823 non-null  object 
dtypes: bool(2), float64(1), object(8)
memory usage: 41.3+ MB


## Step 3 - Check data quality across all three tables

Point: I check shape, duplicates, missing values, and data types for all tables together, not one at a time.

Reason: This gives me one consistent view of data quality, and makes it easy to compare tables directly.

In [11]:
tables = {
    "customers": customers_df,
    "customer_snapshot": customer_snapshot_df,
    "transactions": transactions_df
}

# Basic overview
for name, df in tables.items():
    print(f"\n{'=' * 50}")
    print(f"TABLE: {name}")
    print(f"{'=' * 50}")

    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())

    print("\nMissing values:")
    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })

    display(missing[missing["missing_count"] > 0])

    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))


TABLE: customers
Shape: (2400, 7)
Duplicate rows: 0

Missing values:


,missing_count,missing_percent



Data types:


,dtype
customer_id,object
market,object
join_date,object
age_band,object
account_type,object
acquisition_channel,object
consent_marketing,bool



TABLE: customer_snapshot
Shape: (2400, 14)
Duplicate rows: 0

Missing values:


...

,dtype
customer_id,object
snapshot_date,object
last_app_login,object
balance_eur,float64
overdraft_utilization,float64
support_contacts_30d,int64
declined_txn_30d,int64
salary_in_90d,bool
active_days_30d,int64
txn_count_30d,int64



TABLE: transactions
Shape: (585823, 11)
Duplicate rows: 45

Missing values:


,missing_count,missing_percent
merchant_category,120,0.02



Data types:


,dtype
transaction_id,object
customer_id,object
booked_at,object
direction,object
amount_eur,float64
merchant_category,object
merchant_name,object
channel,object
is_recurring,bool
is_card_present,bool


## Step 3b - Check the target variable

Point: I check how many customers churned versus how many didn't, before doing anything else with this column.

Reason: Class balance decides which metrics will actually be meaningful later. A rare positive class changes how I'd evaluate any future model — I want to know this early, not discover it during modeling.

In [12]:
target_counts = customer_snapshot_df["churned_next_60d"].value_counts(dropna=False)

target_percent = (
    customer_snapshot_df["churned_next_60d"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

display(pd.DataFrame({
    "count": target_counts,
    "percent": target_percent
}))

,count,percent
churned_next_60d,,
False,2219,92.46
True,181,7.54


## Step 3c - Check categorical values

Point: For each categorical column, I look at the unique values and how often each one appears.

Reason: This can reveal inconsistent spelling, extra spaces, or different labels for the same thing. A rare category isn't automatically an error, but it's worth seeing before I decide how to treat it.

In [13]:
categorical_columns = [
    "market",
    "age_band",
    "account_type",
    "acquisition_channel"
]

for col in categorical_columns:
    print(f"\n{col}")
    counts = customers_df[col].value_counts(dropna=False)
    print(counts.to_string())
    display(counts)

for col in ["direction", "channel"]:
    print(f"\n{col}")
    counts = transactions_df[col].value_counts(dropna=False)
    print(counts.to_string())
    display(counts)

# country_code lives in the transactions table, not customers
if "country_code" in transactions_df.columns:
    print("\ncountry_code")
    counts = transactions_df["country_code"].value_counts(dropna=False)
    print(counts.to_string())
    display(counts)


market
market
PL    1063
GB     786
DE     551


,count
market,
PL,1063
GB,786
DE,551



age_band
age_band
35-44    600
25-34    596
45-54    445
55-64    300
18-24    267
65+      192


,count
age_band,
35-44,600
25-34,596
45-54,445
55-64,300
18-24,267
65+,192



account_type
account_type
standard    1363
basic        638
premium      399


...

,count
direction,
debit,573757
credit,12066



channel
channel
card            391104
transfer         78991
direct_debit     72448
cash             43280


,count
channel,
card,391104
transfer,78991
direct_debit,72448
cash,43280



country_code
country_code
PL    251577
GB    181847
DE    128012
FR      6158
ES      6121
IT      6120
US      5988


,count
country_code,
PL,251577
GB,181847
DE,128012
FR,6158
ES,6121
IT,6120
US,5988


## Step 4 - Check that customer IDs are consistent across tables

Point: I check for duplicate IDs, missing IDs, and whether the same customer actually appears in all the tables I expect.

Reason: If a customer exists in transactions but not in the customers table, that's a broken reference — I need to know about it before joining tables together.

In [14]:
# Check customer IDs
print(
    "Duplicate customer IDs in customers:",
    customers_df["customer_id"].duplicated().sum()
)

print(
    "Duplicate customer IDs in snapshot:",
    customer_snapshot_df["customer_id"].duplicated().sum()
)

# Check transaction IDs
print(
    "Duplicate transaction IDs:",
    transactions_df["transaction_id"].duplicated().sum()
)

# Check missing IDs
for name, df in tables.items():
    id_columns = [
        col for col in ["customer_id", "transaction_id"]
        if col in df.columns
    ]

    for col in id_columns:
        print(
            f"{name} — missing {col}:",
            df[col].isna().sum()
        )

Duplicate customer IDs in customers: 0
Duplicate customer IDs in snapshot: 0
Duplicate transaction IDs: 45
customers — missing customer_id: 0
customer_snapshot — missing customer_id: 0
transactions — missing customer_id: 0
transactions — missing transaction_id: 0


In [15]:
customer_ids = set(customers_df["customer_id"].dropna())

snapshot_ids = set(
    customer_snapshot_df["customer_id"].dropna()
)

transaction_customer_ids = set(
    transactions_df["customer_id"].dropna()
)

print(
    "Customers in snapshot but not in customers table:",
    len(snapshot_ids - customer_ids)
)

print(
    "Transaction customers not found in customers table:",
    len(transaction_customer_ids - customer_ids)
)

print(
    "Customers without transactions:",
    len(customer_ids - transaction_customer_ids)
)

Customers in snapshot but not in customers table: 0
Transaction customers not found in customers table: 0
Customers without transactions: 0


In [16]:
print("Customers table:", len(customer_ids))
print("Snapshot table:", len(snapshot_ids))

print("Customers only in customers table:",
      len(customer_ids - snapshot_ids))

print("Customers only in snapshot:",
      len(snapshot_ids - customer_ids))

Customers table: 2400
Snapshot table: 2400
Customers only in customers table: 0
Customers only in snapshot: 0


## Step 5 - Check that dates are valid and make sense

Point: I convert date columns properly, check for invalid dates, and look for dates that don't make logical sense.

Reason: A date stored as text can hide errors. And even a technically valid date can be wrong in context — for example, a customer joining after their own snapshot date shouldn't be possible.

In [17]:
# Convert dates
customers_df["join_date"] = pd.to_datetime(
    customers_df["join_date"],
    errors="coerce"
)

customer_snapshot_df["snapshot_date"] = pd.to_datetime(
    customer_snapshot_df["snapshot_date"],
    errors="coerce"
)

customer_snapshot_df["last_app_login"] = pd.to_datetime(
    customer_snapshot_df["last_app_login"],
    errors="coerce"
)

transactions_df["booked_at"] = pd.to_datetime(
    transactions_df["booked_at"],
    errors="coerce"
)

# Check invalid or missing dates
date_columns = {
    "customers.join_date": customers_df["join_date"],
    "snapshot.snapshot_date": customer_snapshot_df["snapshot_date"],
    "snapshot.last_app_login": customer_snapshot_df["last_app_login"],
    "transactions.booked_at": transactions_df["booked_at"]
}

for name, series in date_columns.items():
    print(f"\n{name}")
    print("Missing or invalid:", series.isna().sum())
    print("Min:", series.min())
    print("Max:", series.max())


customers.join_date
Missing or invalid: 0
Min: 2016-07-02 00:00:00
Max: 2026-03-26 00:00:00

snapshot.snapshot_date
Missing or invalid: 0
Min: 2026-06-30 00:00:00
Max: 2026-06-30 00:00:00

snapshot.last_app_login
Missing or invalid: 0
Min: 2026-01-01 00:00:00
Max: 2026-06-30 00:00:00

transactions.booked_at
Missing or invalid: 0
Min: 2026-01-01 00:00:00
Max: 2026-06-30 00:00:00


In [18]:
# Join date later than snapshot date
customer_dates = customers_df[
    ["customer_id", "join_date"]
].merge(
    customer_snapshot_df[
        ["customer_id", "snapshot_date"]
    ],
    on="customer_id",
    how="inner"
)

joined_after_snapshot = customer_dates[
    customer_dates["join_date"] > customer_dates["snapshot_date"]
]

print("Customers joining after snapshot date:",
      len(joined_after_snapshot))

display(joined_after_snapshot.head())

Customers joining after snapshot date: 0


,customer_id,join_date,snapshot_date


## Step 5b - Check that features respect the snapshot date

Point: I check that features like `txn_count_30d`, `spend_30d`, and `category_diversity_90d` are calculated only from information available on or before the snapshot date.

Reason: A feature name suggests a time window, but it doesn't guarantee it. If any of these features were calculated using data from after the snapshot date, that's data leakage — the model would be trained on information it wouldn't actually have at prediction time.

Decision: Rather than assume this is fine because the column names sound right, I would confirm the exact calculation logic with whoever built the snapshot table. Until confirmed, I treat this as an open question, not a resolved one.

## Step 6 - Check time coverage

Point: I check how many transactions exist per month, across the full six-month period.

Reason: A sudden drop in one month could mean missing data, not just lower customer activity. I need to tell these two apart before trusting any time-based feature.

In [19]:
print("First transaction:", transactions_df["booked_at"].min())
print("Last transaction:", transactions_df["booked_at"].max())

monthly_counts = (
    transactions_df
    .dropna(subset=["booked_at"])
    .set_index("booked_at")
    .resample("MS")
    .size()
    .rename("transaction_count")
)

print(monthly_counts.to_string())
display(monthly_counts)

# Check stability rather than assuming it from a visual scan
mean_count = monthly_counts.mean()
std_count = monthly_counts.std()
cv = std_count / mean_count

print(f"\nMean monthly transactions: {mean_count:.1f}")
print(f"Std dev: {std_count:.1f}")
print(f"Coefficient of variation: {cv:.2%}")

min_month = monthly_counts.idxmin()
max_month = monthly_counts.idxmax()
print(f"\nLowest month: {min_month.date()} ({monthly_counts.min()} transactions)")
print(f"Highest month: {max_month.date()} ({monthly_counts.max()} transactions)")

First transaction: 2026-01-01 00:00:00
Last transaction: 2026-06-30 00:00:00
booked_at
2026-01-01    100120
2026-02-01     90809
2026-03-01    100032
2026-04-01     97244
2026-05-01    100433
2026-06-01     97185
Freq: MS


,transaction_count
booked_at,
2026-01-01,100120
2026-02-01,90809
2026-03-01,100032
2026-04-01,97244
2026-05-01,100433
2026-06-01,97185



Mean monthly transactions: 97637.2
Std dev: 3652.4
Coefficient of variation: 3.74%

Lowest month: 2026-02-01 (90809 transactions)
Highest month: 2026-05-01 (100433 transactions)


## Step 7 - Check numeric ranges and business logic

Point: I look at the summary statistics for key numeric columns, and check for values that shouldn't be possible.

Reason: A statistically valid number can still be wrong in business terms — for example, active days above 30 in a 30-day window.

In [20]:
numeric_columns = [
    "balance_eur",
    "overdraft_utilization",
    "support_contacts_30d",
    "declined_txn_30d",
    "active_days_30d",
    "txn_count_30d",
    "txn_count_prev_30d",
    "spend_30d",
    "category_diversity_90d"
]

display(
    customer_snapshot_df[numeric_columns].describe().T
)

,count,mean,std,min,25%,50%,75%,max
balance_eur,2400.0,4660.346183,6342.349886,-800.00,947.0125,2797.015,5282.32000,44193.85
overdraft_utilization,2400.0,0.229324,0.224501,0.00,0.0670,0.148,0.31225,1.00
support_contacts_30d,2400.0,0.490000,0.794877,0.00,0.0000,0.000,1.00000,6.00
declined_txn_30d,2400.0,1.122083,1.563873,0.00,0.0000,1.000,2.00000,13.00
active_days_30d,2400.0,20.834583,6.159724,2.00,19.0000,23.000,25.00000,30.00
txn_count_30d,2400.0,39.657500,16.019065,2.00,32.0000,42.000,51.00000,76.00
txn_count_prev_30d,2400.0,39.692917,15.888664,2.00,32.0000,43.000,51.00000,76.00
spend_30d,2400.0,4999.289167,2525.942730,50.43,3457.5225,4869.045,6522.39000,15520.93
category_diversity_90d,2400.0,12.613750,1.361494,5.00,12.0000,13.000,13.00000,14.00


In [21]:
# Potentially invalid values
checks = {
    "negative active days":
        customer_snapshot_df["active_days_30d"] < 0,

    "active days above 30":
        customer_snapshot_df["active_days_30d"] > 30,

    "negative transaction count":
        customer_snapshot_df["txn_count_30d"] < 0,

    "negative previous transaction count":
        customer_snapshot_df["txn_count_prev_30d"] < 0,

    "negative declined transaction count":
        customer_snapshot_df["declined_txn_30d"] < 0,

    "negative support contacts":
        customer_snapshot_df["support_contacts_30d"] < 0,

    "negative category diversity":
        customer_snapshot_df["category_diversity_90d"] < 0,

    "negative overdraft utilization":
        customer_snapshot_df["overdraft_utilization"] < 0
}

for description, condition in checks.items():
    print(description + ":", condition.sum())

negative active days: 0
active days above 30: 0
negative transaction count: 0
negative previous transaction count: 0
negative declined transaction count: 0
negative support contacts: 0
negative category diversity: 0
negative overdraft utilization: 0


## Step 8 - Look closely at duplicate transaction IDs

Point: I found 45 repeated `transaction_id` values earlier. Before deciding what to do about them, I need to see what's actually repeating.

Reason: A repeated ID could be a true duplicate record, or it could reflect something about how the source system generates or processes transactions. I don't remove anything until I understand which one it is.

In [22]:
duplicate_transactions = transactions_df[
    transactions_df["transaction_id"].duplicated(keep=False)
].sort_values("transaction_id")

display(duplicate_transactions)

exact_dupes = transactions_df.duplicated().sum()
print("Exact duplicate rows (DataFrame.duplicated()):", exact_dupes)
print("These are full-row duplicates, not just repeated IDs — this still does not tell me")
print("whether they come from a re-ingestion of the same file, or a source-system behavior.")

# Check whether rows sharing a transaction_id also match on other key fields
match_check = (
    duplicate_transactions
    .groupby("transaction_id")[["customer_id", "booked_at", "amount_eur"]]
    .nunique()
)

print("\nFor each duplicated transaction_id, number of distinct values per field (1 = fully consistent):")
print(match_check.to_string())
display(match_check)

,transaction_id,customer_id,booked_at,direction,amount_eur,merchant_category,merchant_name,channel,is_recurring,is_card_present,country_code
478972,T000007685,C000033,2026-05-29,debit,-14.18,restaurants,Restaurants_21,card,False,True,DE
585788,T000007685,C000033,2026-05-29,debit,-14.18,restaurants,Restaurants_21,card,False,True,DE
381958,T000011177,C000047,2026-04-29,debit,-79.30,groceries,Groceries_02,card,False,False,DE
585802,T000011177,C000047,2026-04-29,debit,-79.30,groceries,Groceries_02,card,False,False,DE
76630,T000012041,C000050,2026-01-25,debit,-102.06,restaurants,Restaurants_18,transfer,False,False,FR
...,...,...,...,...,...,...,...,...,...,...,...
585799,T000541947,C002222,2026-06-22,debit,-62.02,fuel,Fuel_16,card,False,True,DE
585812,T000569161,C002333,2026-01-06,debit,-161.35,entertainment,Entertainment_06,card,False,True,GB
19541,T000569161,C002333,2026-01-06,debit,-161.35,entertainment,Entertainment_06,card,False,True,GB
348957,T000582428,C002386,2026-04-18,debit,-52.79,shopping,Shopping_03,card,False,True,GB


Exact duplicate rows (DataFrame.duplicated()): 45
These are full-row duplicates, not just repeated IDs — this still does not tell me
whether they come from a re-ingestion of the same file, or a source-system behavior.

For each duplicated transaction_id, number of distinct values per field (1 = fully consistent):
                customer_id  booked_at  amount_eur
transaction_id                                    
T000007685                1          1           1
T000011177                1          1           1
T000012041                1          1           1
T000012758                1          1           1
T000032920                1          1           1
T000037586                1          1           1
T000056732                1          1           1
T000083621                1          1           1
T000095501                1          1           1
T000098919                1          1           1
T000139960                1          1           1
T000141874            

,customer_id,booked_at,amount_eur
transaction_id,,,
T000007685,1,1,1
T000011177,1,1,1
T000012041,1,1,1
T000012758,1,1,1
T000032920,1,1,1
T000037586,1,1,1
T000056732,1,1,1
T000083621,1,1,1
T000095501,1,1,1


## Step 9 - Check transaction amounts for outliers

Point: I check the distribution of transaction amounts, and flag values far outside the normal range using the IQR rule.

Reason: The IQR rule flags values far from the center of the distribution — it does not mean those values are errors. With skewed monetary data, a meaningful share of real transactions can fall outside this range. I look at them separately by direction, since debits and credits are economically different and shouldn't be judged against one mixed distribution.

In [23]:
print("Transaction amount summary:")
display(transactions_df["amount_eur"].describe())

print("\nAmount summary by direction:")
display(transactions_df.groupby("direction")["amount_eur"].describe())

q1 = transactions_df["amount_eur"].quantile(0.25)
q3 = transactions_df["amount_eur"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = transactions_df[
    (transactions_df["amount_eur"] < lower_bound) | (transactions_df["amount_eur"] > upper_bound)
]
print(f"\nPotential outliers flagged by the IQR rule: {len(outliers)} ({len(outliers)/len(transactions_df)*100:.2f}%)")

print("\nSmallest amounts:")
display(transactions_df.sort_values("amount_eur").head(10))

print("\nLargest amounts:")
display(transactions_df.sort_values("amount_eur", ascending=False).head(10))

Transaction amount summary:


,amount_eur
count,585823.000000
mean,-50.508011
std,638.073958
min,-9759.980000
25%,-83.600000
50%,-39.260000
75%,-20.240000
max,9974.260000



Amount summary by direction:


,count,mean,std,min,25%,50%,75%,max
direction,,,,,,,,
credit,12066.0,3526.340796,1700.043526,1391.67,2488.42,2982.745,3767.07,9974.26
debit,573757.0,-125.728457,283.223138,-9759.98,-85.43,-40.280,-21.19,-1.25



Potential outliers flagged by the IQR rule: 86501 (14.77%)

Smallest amounts:


,transaction_id,customer_id,booked_at,direction,amount_eur,merchant_category,merchant_name,channel,is_recurring,is_card_present,country_code
264917,T000106256,C000425,2026-03-24,debit,-9759.98,rent,Rent_34,card,True,True,DE
307371,T000585029,C002398,2026-04-05,debit,-8687.03,rent,Rent_27,card,False,True,DE
42546,T000157264,C000639,2026-01-14,debit,-8411.35,rent,Rent_13,card,True,False,PL
116582,T000551049,C002259,2026-02-05,debit,-8032.41,rent,Rent_27,card,False,True,PL
331855,T000412056,C001691,2026-04-13,debit,-7756.80,rent,Rent_13,card,True,True,GB
423113,T000433343,C001777,2026-05-11,debit,-7711.31,rent,Rent_30,direct_debit,True,False,PL
256376,T000265161,C001083,2026-03-21,debit,-6676.27,rent,Rent_29,card,False,True,PL
26986,T000177928,C000720,2026-01-09,debit,-6081.98,rent,Rent_06,card,True,True,PL
126277,T000030369,C000121,2026-02-09,debit,-5831.12,rent,Rent_01,transfer,True,False,PL
529370,T000365515,C001497,2026-06-13,debit,-5802.76,rent,Rent_27,card,True,False,PL



Largest amounts:


,transaction_id,customer_id,booked_at,direction,amount_eur,merchant_category,merchant_name,channel,is_recurring,is_card_present,country_code
179002,T000322654,C001323,2026-02-25,credit,9974.26,income,EMPLOYER_TRANSFER,transfer,True,False,PL
180605,T000559794,C002292,2026-02-25,credit,9861.69,income,EMPLOYER_TRANSFER,transfer,True,False,PL
314,T000050626,C000199,2026-01-01,credit,9703.18,income,EMPLOYER_TRANSFER,transfer,True,False,GB
492090,T000545224,C002234,2026-06-01,credit,9482.70,income,EMPLOYER_TRANSFER,transfer,True,False,GB
379049,T000149835,C000607,2026-04-28,credit,9377.67,income,EMPLOYER_TRANSFER,transfer,True,False,PL
369842,T000274575,C001121,2026-04-25,credit,9353.43,income,EMPLOYER_TRANSFER,transfer,True,False,GB
191211,T000046579,C000184,2026-03-01,credit,9336.52,income,EMPLOYER_TRANSFER,transfer,True,False,PL
278905,T000196777,C000797,2026-03-28,credit,9311.03,income,EMPLOYER_TRANSFER,transfer,True,False,DE
490232,T000252821,C001031,2026-06-01,credit,9284.46,income,EMPLOYER_TRANSFER,transfer,True,False,DE
268470,T000161619,C000658,2026-03-25,credit,9268.59,income,EMPLOYER_TRANSFER,transfer,True,False,DE


## Step 10 - Check missing merchant categories

Point: I check how many transactions have no merchant category, and look at what those transactions have in common.

Reason: A missing merchant category could mean the merchant name wasn't matched to a known category — but with only 120 cases (about 0.02% of transactions), I don't yet have enough evidence to say that's the cause. This needs more investigation before I treat it as a confirmed explanation rather than one possibility.

In [24]:
missing_category = transactions_df["merchant_category"].isna().sum()
print("Missing merchant_category:", missing_category,
      f"({missing_category/len(transactions_df)*100:.3f}% of transactions)")

display(transactions_df[transactions_df["merchant_category"].isna()].head(10))

print("\nChannels for transactions with missing category:")
display(transactions_df[transactions_df["merchant_category"].isna()]["channel"].value_counts())

Missing merchant_category: 120 (0.020% of transactions)


,transaction_id,customer_id,booked_at,direction,amount_eur,merchant_category,merchant_name,channel,is_recurring,is_card_present,country_code
13002,T000539454,C002212,2026-01-04,debit,-8.13,NaN,Transport_01,transfer,False,False,PL
17321,T000156335,C000636,2026-01-06,debit,-18.25,NaN,Restaurants_12,card,False,True,DE
24205,T000248298,C001015,2026-01-08,debit,-41.85,NaN,Entertainment_08,transfer,False,False,DE
26191,T000035795,C000143,2026-01-09,debit,-54.66,NaN,Groceries_23,card,False,True,PL
28612,T000488815,C002003,2026-01-09,debit,-18.95,NaN,Entertainment_09,card,False,True,PL
35484,T000013981,C000058,2026-01-12,debit,-21.11,NaN,Transport_08,direct_debit,False,False,GB
36671,T000239472,C000976,2026-01-12,debit,-87.90,NaN,Utilities_24,direct_debit,True,False,GB
38884,T000070580,C000282,2026-01-13,debit,-315.20,NaN,Travel_15,transfer,False,False,GB
46277,T000257441,C001049,2026-01-15,debit,-187.74,NaN,Insurance_33,card,False,False,PL
49113,T000172604,C000699,2026-01-16,debit,-250.80,NaN,Travel_24,card,False,False,PL



Channels for transactions with missing category:


,count
channel,
card,76
transfer,18
direct_debit,15
cash,11


## Appendix - Merchant Matching (Proof of Concept)

This section is optional exploration, separate from the core data quality review above. It shows my understanding of the merchant-matching problem, not a production-ready fix — I have not filled in any missing values in the main tables.

In [25]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.5 MB/s eta 0:00:00


### Check for ambiguous merchant names first

Point: Before building a name-to-category reference, I check whether any merchant name is already linked to more than one category.

Reason: If I build the reference by just dropping duplicate names, an ambiguous merchant would silently keep only one of its categories — possibly the wrong one for a given transaction. I need to see how big this problem is before deciding how to handle it.

In [26]:
category_counts = (
    transactions_df.dropna(subset=["merchant_category"])
    .groupby("merchant_name")["merchant_category"]
    .nunique()
)

ambiguous_merchants = category_counts[category_counts > 1]

print("Merchant names with multiple categories:", len(ambiguous_merchants))
display(ambiguous_merchants.head(20))

Merchant names with multiple categories: 0


,merchant_category
merchant_name,


### Normalize merchant names before matching

Point: I lowercase, strip, and remove punctuation from merchant names before comparing them.

Reason: Exact comparison is case- and whitespace-sensitive, so the same merchant can appear to be two different ones. I stay cautious here — aggressive normalization could also merge names that are genuinely different companies, so this output still needs a manual look, not blind trust.

In [27]:
import re

def normalize_merchant(name):
    if pd.isna(name):
        return None
    name = name.lower().strip()
    # \w is Unicode-aware in Python 3, so this keeps accented letters and
    # non-Latin scripts intact instead of stripping them like an ASCII-only rule would.
    name = re.sub(r"[^\w\s]", " ", name, flags=re.UNICODE)
    name = re.sub(r"\s+", " ", name)
    return name

transactions_df["merchant_name_normalized"] = transactions_df["merchant_name"].apply(normalize_merchant)

### Match missing categories with caution

Point: For merchants with a missing category, I search for the closest known merchant name and only accept the match above a similarity threshold.

Reason: The threshold of 85 used below is a starting assumption, not a validated cutoff — I would manually review a sample of matches at different scores before trusting it. A fuzzy match should never silently become a confirmed category; low-confidence or ambiguous matches should be left unresolved rather than guessed.

In [28]:
from rapidfuzz import process, fuzz

reference = (
    transactions_df.dropna(subset=["merchant_category"])
    .drop_duplicates(subset=["merchant_name_normalized"])
    .set_index("merchant_name_normalized")["merchant_category"]
    .to_dict()
)
reference_names = list(reference.keys())

def match_category(merchant_name_normalized, threshold=85):
    if merchant_name_normalized in reference:
        return reference[merchant_name_normalized], 100
    best_match, score, _ = process.extractOne(
        merchant_name_normalized, reference_names, scorer=fuzz.ratio
    )
    if score >= threshold:
        return reference[best_match], score
    return None, score

missing_rows = transactions_df[transactions_df["merchant_category"].isna()].copy()

missing_rows[["matched_category", "match_score"]] = missing_rows["merchant_name_normalized"].apply(
    lambda name: pd.Series(match_category(name))
)

print("Match score distribution:")
print(missing_rows["match_score"].value_counts().to_string())
display(missing_rows["match_score"].value_counts())

print("\nUnmatched (no category found above threshold):", missing_rows["matched_category"].isna().sum())

# The threshold of 85 is an assumption, not a validated cutoff. Before trusting any of
# these matches, I would manually review a sample across different score bands.
print("\nManual review sample, by score band (not yet validated by a human):")

score_bins = [(85, 90), (90, 95), (95, 100)]
for low, high in score_bins:
    band = missing_rows[
        (missing_rows["match_score"] >= low) & (missing_rows["match_score"] < high)
    ]
    print(f"\nScore {low}-{high}: {len(band)} rows")
    sample = band[["merchant_name", "matched_category", "match_score"]].head(5)
    print(sample.to_string())
    display(sample)

perfect = missing_rows[missing_rows["match_score"] == 100]
print(f"\nExact matches after normalization (score 100): {len(perfect)} rows")
display(perfect[["merchant_name", "matched_category", "match_score"]].head(10))

Match score distribution:
match_score
100    120


,count
match_score,
100,120



Unmatched (no category found above threshold): 0

Manual review sample, by score band (not yet validated by a human):

Score 85-90: 0 rows
Empty DataFrame
Columns: [merchant_name, matched_category, match_score]
Index: []


,merchant_name,matched_category,match_score



Score 90-95: 0 rows
Empty DataFrame
Columns: [merchant_name, matched_category, match_score]
Index: []


,merchant_name,matched_category,match_score



Score 95-100: 0 rows
Empty DataFrame
Columns: [merchant_name, matched_category, match_score]
Index: []


,merchant_name,matched_category,match_score



Exact matches after normalization (score 100): 120 rows


,merchant_name,matched_category,match_score
13002,Transport_01,transport,100
17321,Restaurants_12,restaurants,100
24205,Entertainment_08,entertainment,100
26191,Groceries_23,groceries,100
28612,Entertainment_09,entertainment,100
35484,Transport_08,transport,100
36671,Utilities_24,utilities,100
38884,Travel_15,travel,100
46277,Insurance_33,insurance,100
49113,Travel_24,travel,100


## Summary of data quality findings

With this understanding of the data, my next step would be feature engineering — starting with behavioural features like spend volatility, income stability, and balance trajectory, using the findings below to decide how to handle sparse, ambiguous, or unusual cases.

In [29]:
# Pull real, already-computed values into the summary rather than hardcoding claims
churn_summary = ", ".join(
    f"{idx}: {cnt} ({pct}%)"
    for idx, cnt, pct in zip(target_counts.index, target_counts.values, target_percent.values)
)

join_after_snapshot_result = (
    f"{len(joined_after_snapshot)} customers found with join date after snapshot date"
)

monthly_volume_result = (
    f"Mean {mean_count:.0f}/month, CV {cv:.1%}, "
    f"lowest {monthly_counts.min()} ({min_month.date()}), highest {monthly_counts.max()} ({max_month.date()})"
)

findings = pd.DataFrame([
    {"Check": "Table structure & types", "Result": "Load cleanly, consistent dtypes after conversion",
     "Interpretation": "No structural issues found", "Action": "None needed"},
    {"Check": "Customer ID consistency", "Result": "IDs consistent across customers, snapshot, transactions",
     "Interpretation": "No major broken references", "Action": "None needed"},
    {"Check": "Join date vs snapshot date", "Result": join_after_snapshot_result,
     "Interpretation": "No cases found in this run" if len(joined_after_snapshot) == 0 else "Logically invalid cases present",
     "Action": "None needed" if len(joined_after_snapshot) == 0 else "Confirm with data provider before use"},
    {"Check": "Feature temporal availability", "Result": "Not yet confirmed against calculation logic",
     "Interpretation": "Possible leakage risk, unverified", "Action": "Confirm calculation windows with the team before modeling"},
    {"Check": "Transaction volume by month", "Result": monthly_volume_result,
     "Interpretation": "Quantified via coefficient of variation, not just a visual scan",
     "Action": "Review flagged low/high months if CV is high"},
    {"Check": "Snapshot business rules", "Result": "No invalid negative values found",
     "Interpretation": "Numeric fields look valid", "Action": "None needed"},
    {"Check": "Duplicate transaction_id", "Result": f"{exact_dupes} exact duplicate rows (DataFrame.duplicated()); field-level consistency checked",
     "Interpretation": "Cause not yet confirmed — re-ingestion vs. source behavior both possible",
     "Action": "Investigate with source/ingestion team before deciding whether to deduplicate"},
    {"Check": "Transaction amount outliers", "Result": f"{len(outliers)} rows flagged by IQR rule ({len(outliers)/len(transactions_df)*100:.1f}%), reviewed by direction",
     "Interpretation": "Not necessarily errors; skewed by nature of monetary data", "Action": "Review by direction before transforming"},
    {"Check": "Missing merchant_category", "Result": f"{missing_category} transactions ({missing_category/len(transactions_df)*100:.3f}%)",
     "Interpretation": "Possible merchant-matching gap — one hypothesis, not yet confirmed", "Action": "Investigate further (see Appendix)"},
    {"Check": "Merchant-matching proof of concept", "Result": f"{missing_rows['matched_category'].notna().sum()} matched, {missing_rows['matched_category'].isna().sum()} unmatched at threshold 85",
     "Interpretation": "Threshold not yet validated by manual review — see score-band samples above",
     "Action": "Manually validate a sample per score band before trusting any match"},
    {"Check": "Target variable balance", "Result": churn_summary,
     "Interpretation": "Shapes which metrics will matter later (e.g. accuracy vs. ROC-AUC/precision/recall)",
     "Action": "Choose evaluation metrics accordingly during modeling"},
])

pd.set_option("display.max_colwidth", None)

styled = (
    findings.style
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
    .hide(axis="index")
)

display(styled)

Check,Result,Interpretation,Action
Table structure & types,"Load cleanly, consistent dtypes after conversion",No structural issues found,None needed
Customer ID consistency,"IDs consistent across customers, snapshot, transactions",No major broken references,None needed
Join date vs snapshot date,0 customers found with join date after snapshot date,No cases found in this run,None needed
Feature temporal availability,Not yet confirmed against calculation logic,"Possible leakage risk, unverified",Confirm calculation windows with the team before modeling
Transaction volume by month,"Mean 97637/month, CV 3.7%, lowest 90809 (2026-02-01), highest 100433 (2026-05-01)","Quantified via coefficient of variation, not just a visual scan",Review flagged low/high months if CV is high
Snapshot business rules,No invalid negative values found,Numeric fields look valid,None needed
Duplicate transaction_id,45 exact duplicate rows (DataFrame.duplicated()); field-level consistency checked,Cause not yet confirmed — re-ingestion vs. source behavior both possible,Investigate with source/ingestion team before deciding whether to deduplicate
Transaction amount outliers,"86501 rows flagged by IQR rule (14.8%), reviewed by direction",Not necessarily errors; skewed by nature of monetary data,Review by direction before transforming
Missing merchant_category,120 transactions (0.020%),"Possible merchant-matching gap — one hypothesis, not yet confirmed",Investigate further (see Appendix)
Merchant-matching proof of concept,"120 matched, 0 unmatched at threshold 85",Threshold not yet validated by manual review — see score-band samples above,Manually validate a sample per score band before trusting any match
